In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

## ** TRANSFORMATIONS **

In [0]:
from typing import List
from pyspark.sql import DataFrame
from pyspark.sql.window import Window
from pyspark.sql.functions import *
from delta.tables import DeltaTable

In [0]:
df_cust = spark.read.table("uber_dbt.bronze.customers")
display(df_cust)

customer_id,first_name,last_name,email,phone_number,city,signup_date,last_updated_timestamp
1,Daniel,Reed,azimmerman@ramirez-nelson.com,811-724-1080,Tiffanyview,2023-12-25,2025-09-15T15:21:05.000Z
2,Jonathan,Hansen,williammiller@serrano-jones.com,632-190-4027x0198,Baldwinburgh,2021-01-23,2025-09-19T20:19:41.000Z
3,Samuel,Rodriguez,troy07@carrillo-webb.info,676.094.0716,New Ashley,2025-05-31,2025-09-13T03:45:53.000Z
4,Victor,Sanchez,jonthompson@thomas.biz,960-929-2694,Stephanieton,2024-08-31,2025-08-23T00:38:03.000Z
5,Sabrina,Black,patricknewman@williams.biz,+1-621-699-9458x79470,South Christopherport,2023-05-18,2025-08-20T16:08:38.000Z
6,Melissa,Blair,courtney29@gmail.com,(332)469-2812x91807,East Pamela,2025-09-17,2025-08-24T09:44:20.000Z
7,Jacqueline,Williams,robertmoore@gmail.com,7312603248,Fritzmouth,2023-05-29,2025-08-28T19:40:31.000Z
8,Christina,Arnold,parkerbridget@yahoo.com,(878)682-1357,North Sarah,2023-09-21,2025-09-06T20:36:02.000Z
9,Lisa,Shelton,cody49@johnson.org,9349468560,South Kyleview,2022-12-13,2025-09-08T14:57:58.000Z
10,Megan,Dean,vritter@yahoo.com,001-967-061-8100x482,Port Jesse,2023-03-12,2025-09-07T23:05:32.000Z


In [0]:
df_cust = df_cust.withColumn("domain",split(col("email"),"@").getItem(1))
display(df_cust)

customer_id,first_name,last_name,email,phone_number,city,signup_date,last_updated_timestamp,domain
1,Daniel,Reed,azimmerman@ramirez-nelson.com,811-724-1080,Tiffanyview,2023-12-25,2025-09-15T15:21:05.000Z,ramirez-nelson.com
2,Jonathan,Hansen,williammiller@serrano-jones.com,632-190-4027x0198,Baldwinburgh,2021-01-23,2025-09-19T20:19:41.000Z,serrano-jones.com
3,Samuel,Rodriguez,troy07@carrillo-webb.info,676.094.0716,New Ashley,2025-05-31,2025-09-13T03:45:53.000Z,carrillo-webb.info
4,Victor,Sanchez,jonthompson@thomas.biz,960-929-2694,Stephanieton,2024-08-31,2025-08-23T00:38:03.000Z,thomas.biz
5,Sabrina,Black,patricknewman@williams.biz,+1-621-699-9458x79470,South Christopherport,2023-05-18,2025-08-20T16:08:38.000Z,williams.biz
6,Melissa,Blair,courtney29@gmail.com,(332)469-2812x91807,East Pamela,2025-09-17,2025-08-24T09:44:20.000Z,gmail.com
7,Jacqueline,Williams,robertmoore@gmail.com,7312603248,Fritzmouth,2023-05-29,2025-08-28T19:40:31.000Z,gmail.com
8,Christina,Arnold,parkerbridget@yahoo.com,(878)682-1357,North Sarah,2023-09-21,2025-09-06T20:36:02.000Z,yahoo.com
9,Lisa,Shelton,cody49@johnson.org,9349468560,South Kyleview,2022-12-13,2025-09-08T14:57:58.000Z,johnson.org
10,Megan,Dean,vritter@yahoo.com,001-967-061-8100x482,Port Jesse,2023-03-12,2025-09-07T23:05:32.000Z,yahoo.com


In [0]:
df_cust = df_cust.withColumn("phone_number",regexp_replace("phone_number",r"[^0-9]",""))
display(df_cust)

customer_id,first_name,last_name,email,phone_number,city,signup_date,last_updated_timestamp,domain
1,Daniel,Reed,azimmerman@ramirez-nelson.com,8117241080,Tiffanyview,2023-12-25,2025-09-15T15:21:05.000Z,ramirez-nelson.com
2,Jonathan,Hansen,williammiller@serrano-jones.com,63219040270198,Baldwinburgh,2021-01-23,2025-09-19T20:19:41.000Z,serrano-jones.com
3,Samuel,Rodriguez,troy07@carrillo-webb.info,6760940716,New Ashley,2025-05-31,2025-09-13T03:45:53.000Z,carrillo-webb.info
4,Victor,Sanchez,jonthompson@thomas.biz,9609292694,Stephanieton,2024-08-31,2025-08-23T00:38:03.000Z,thomas.biz
5,Sabrina,Black,patricknewman@williams.biz,1621699945879470,South Christopherport,2023-05-18,2025-08-20T16:08:38.000Z,williams.biz
6,Melissa,Blair,courtney29@gmail.com,332469281291807,East Pamela,2025-09-17,2025-08-24T09:44:20.000Z,gmail.com
7,Jacqueline,Williams,robertmoore@gmail.com,7312603248,Fritzmouth,2023-05-29,2025-08-28T19:40:31.000Z,gmail.com
8,Christina,Arnold,parkerbridget@yahoo.com,8786821357,North Sarah,2023-09-21,2025-09-06T20:36:02.000Z,yahoo.com
9,Lisa,Shelton,cody49@johnson.org,9349468560,South Kyleview,2022-12-13,2025-09-08T14:57:58.000Z,johnson.org
10,Megan,Dean,vritter@yahoo.com,0019670618100482,Port Jesse,2023-03-12,2025-09-07T23:05:32.000Z,yahoo.com


In [0]:
df_cust = df_cust.withColumn("full_name",concat_ws(" ",col("first_name"),col("last_name")))
df_cust = df_cust.drop("first_name","last_name")
display(df_cust)

customer_id,email,phone_number,city,signup_date,last_updated_timestamp,domain,full_name
1,azimmerman@ramirez-nelson.com,8117241080,Tiffanyview,2023-12-25,2025-09-15T15:21:05.000Z,ramirez-nelson.com,Daniel Reed
2,williammiller@serrano-jones.com,63219040270198,Baldwinburgh,2021-01-23,2025-09-19T20:19:41.000Z,serrano-jones.com,Jonathan Hansen
3,troy07@carrillo-webb.info,6760940716,New Ashley,2025-05-31,2025-09-13T03:45:53.000Z,carrillo-webb.info,Samuel Rodriguez
4,jonthompson@thomas.biz,9609292694,Stephanieton,2024-08-31,2025-08-23T00:38:03.000Z,thomas.biz,Victor Sanchez
5,patricknewman@williams.biz,1621699945879470,South Christopherport,2023-05-18,2025-08-20T16:08:38.000Z,williams.biz,Sabrina Black
6,courtney29@gmail.com,332469281291807,East Pamela,2025-09-17,2025-08-24T09:44:20.000Z,gmail.com,Melissa Blair
7,robertmoore@gmail.com,7312603248,Fritzmouth,2023-05-29,2025-08-28T19:40:31.000Z,gmail.com,Jacqueline Williams
8,parkerbridget@yahoo.com,8786821357,North Sarah,2023-09-21,2025-09-06T20:36:02.000Z,yahoo.com,Christina Arnold
9,cody49@johnson.org,9349468560,South Kyleview,2022-12-13,2025-09-08T14:57:58.000Z,johnson.org,Lisa Shelton
10,vritter@yahoo.com,0019670618100482,Port Jesse,2023-03-12,2025-09-07T23:05:32.000Z,yahoo.com,Megan Dean


### _BUILDING A DEDUPLICATION FUNCTION FOR DYNAMIC DEDUP_

In [0]:
class transformations:

    def dedup(self,df:DataFrame, dedup_cols:List,cdc:str):

        df = df.withColumn("dedup_key",concat(*dedup_cols))
        df = df.withColumn("dedupCounts",row_number().over(Window.partitionBy("dedup_key").orderBy(desc(cdc))))
        df = df.filter(col("dedupCounts") == 1)
        df = df.drop("dedup_key","dedupCounts")
        return df
    
    def process_timestamp(self,df):
        df = df.withColumn("process_date",current_timestamp())
        return df
    
    def upsert(self,df,key_cols,table,cdc):
        merge_condition = ' AND '.join([f"s.{i} = t.{i}" for i in key_cols])
        dlt_obj = DeltaTable.forName(spark,f"uber_dbt.silver.{table}")
        dlt_obj.alias("t").merge(
            df.alias("s"),
            merge_condition
        ).whenMatchedUpdateAll(condition = f"s.{cdc} > t.{cdc}").whenNotMatchedInsertAll().execute()
        return
    
   

## _DEDUPLICATION_

In [0]:
cust_obj = transformations()
df_cust_trans = cust_obj.dedup(df_cust,["customer_id"],"last_updated_timestamp")
display (df_cust)


customer_id,email,phone_number,city,signup_date,last_updated_timestamp,domain,full_name
1,azimmerman@ramirez-nelson.com,8117241080,Tiffanyview,2023-12-25,2025-09-15T15:21:05.000Z,ramirez-nelson.com,Daniel Reed
2,williammiller@serrano-jones.com,63219040270198,Baldwinburgh,2021-01-23,2025-09-19T20:19:41.000Z,serrano-jones.com,Jonathan Hansen
3,troy07@carrillo-webb.info,6760940716,New Ashley,2025-05-31,2025-09-13T03:45:53.000Z,carrillo-webb.info,Samuel Rodriguez
4,jonthompson@thomas.biz,9609292694,Stephanieton,2024-08-31,2025-08-23T00:38:03.000Z,thomas.biz,Victor Sanchez
5,patricknewman@williams.biz,1621699945879470,South Christopherport,2023-05-18,2025-08-20T16:08:38.000Z,williams.biz,Sabrina Black
6,courtney29@gmail.com,332469281291807,East Pamela,2025-09-17,2025-08-24T09:44:20.000Z,gmail.com,Melissa Blair
7,robertmoore@gmail.com,7312603248,Fritzmouth,2023-05-29,2025-08-28T19:40:31.000Z,gmail.com,Jacqueline Williams
8,parkerbridget@yahoo.com,8786821357,North Sarah,2023-09-21,2025-09-06T20:36:02.000Z,yahoo.com,Christina Arnold
9,cody49@johnson.org,9349468560,South Kyleview,2022-12-13,2025-09-08T14:57:58.000Z,johnson.org,Lisa Shelton
10,vritter@yahoo.com,0019670618100482,Port Jesse,2023-03-12,2025-09-07T23:05:32.000Z,yahoo.com,Megan Dean


## _ADDING PROCESS DATE_

In [0]:
df_cust_trans = cust_obj.process_timestamp(df_cust_trans)
display (df_cust_trans)

customer_id,email,phone_number,city,signup_date,last_updated_timestamp,domain,full_name,process_date
1,azimmerman@ramirez-nelson.com,8117241080,Tiffanyview,2023-12-25,2025-09-15T15:21:05.000Z,ramirez-nelson.com,Daniel Reed,2026-09-16T16:26:33.615Z
10,vritter@yahoo.com,0019670618100482,Port Jesse,2023-03-12,2025-09-07T23:05:32.000Z,yahoo.com,Megan Dean,2026-09-16T16:26:33.615Z
100,newmanmelanie@flynn-ross.org,0010575302069,North Charlestown,2022-06-22,2025-09-14T22:47:28.000Z,flynn-ross.org,Gary Barnes,2026-09-16T16:26:33.615Z
101,hannahwhite@hotmail.com,8882340212190,Lisachester,2020-09-24,2025-09-10T22:45:30.000Z,hotmail.com,Stacy Thomas,2026-09-16T16:26:33.615Z
102,cunninghamjessica@yahoo.com,0010591315078,Harrisbury,2023-07-29,2025-09-19T21:57:29.000Z,yahoo.com,Alan Rogers,2026-09-16T16:26:33.615Z
103,sherry55@hernandez.com,12586465512757,Port Michael,2022-06-18,2025-09-09T06:06:57.000Z,hernandez.com,Stephen Sanders,2026-09-16T16:26:33.615Z
104,moralessandy@arnold-robinson.com,0018634267566120,Lake Kelly,2020-11-07,2025-09-17T02:22:22.000Z,arnold-robinson.com,Duane Bennett,2026-09-16T16:26:33.615Z
105,matthewsbenjamin@lloyd.com,87710366965279,South Wyattfort,2021-04-11,2025-08-25T11:18:42.000Z,lloyd.com,Jasmin Patterson,2026-09-16T16:26:33.615Z
106,tyleryoung@gross.org,41127166452578,Seanview,2024-01-05,2025-08-25T04:33:25.000Z,gross.org,Kara Williams,2026-09-16T16:26:33.615Z
107,brandyphillips@hotmail.com,17891625036,Melissabury,2021-06-15,2025-09-14T13:56:22.000Z,hotmail.com,Martin Williams,2026-09-16T16:26:33.615Z


## _UPSERT_

In [0]:

if not spark.catalog.tableExists("uber_dbt.silver.customers"):
  df_cust_trans.write.mode("append").saveAsTable("uber_dbt.silver.customers")

else:
    cust_obj.upsert(df_cust_trans,["customer_id"],"customers","last_updated_timestamp")

In [0]:
%sql
SELECT COUNT(*) AS total from uber_dbt.silver.customers

total
200


# **DRIVERS**

In [0]:
df_drivers = spark.read.table("uber_dbt.bronze.drivers")
display(df_drivers)


driver_id,first_name,last_name,phone_number,vehicle_id,driver_rating,city,last_updated_timestamp
1,Latasha,Lopez,262-924-2955x590,1,4.7,East Dorothy,2025-08-25T06:36:26.000Z
2,Alan,Wiley,0967969634,2,3.98,West Susan,2025-09-14T00:44:57.000Z
3,James,Taylor,424-614-1847,3,3.66,Mcintoshton,2025-08-26T22:28:17.000Z
4,Theresa,Benson,617-017-0101x91777,4,3.86,North Courtneychester,2025-09-01T11:40:55.000Z
5,Karen,Jensen,611-060-5683,5,4.87,Brownburgh,2025-09-04T16:35:04.000Z
6,Debra,Smith,556.480.9096x439,6,4.26,Port Williamland,2025-08-31T14:31:37.000Z
7,Justin,Peters,+1-798-568-6952x9778,7,3.67,West Erinborough,2025-09-17T05:57:45.000Z
8,Todd,Young,706.321.8390x08097,8,4.9,Lake Stephen,2025-08-26T06:45:51.000Z
9,Mary,Young,(172)791-0504x5499,9,4.5,West Lindsey,2025-08-27T13:04:32.000Z
10,Jacob,Mack,(509)613-4480,10,4.04,Lauraland,2025-09-09T05:50:23.000Z


In [0]:
df_drivers = df_drivers.withColumn("full_name",concat_ws(" ",col("first_name"),col("last_name")))
df_drivers = df_drivers.drop("first_name","last_name")


In [0]:
df_drivers = df_drivers.withColumn("phone_number",regexp_replace("phone_number",r"[^0-9]",""))


In [0]:
display (df_drivers)

driver_id,phone_number,vehicle_id,driver_rating,city,last_updated_timestamp,full_name
1,2629242955590,1,4.7,East Dorothy,2025-08-25T06:36:26.000Z,Latasha Lopez
2,0967969634,2,3.98,West Susan,2025-09-14T00:44:57.000Z,Alan Wiley
3,4246141847,3,3.66,Mcintoshton,2025-08-26T22:28:17.000Z,James Taylor
4,617017010191777,4,3.86,North Courtneychester,2025-09-01T11:40:55.000Z,Theresa Benson
5,6110605683,5,4.87,Brownburgh,2025-09-04T16:35:04.000Z,Karen Jensen
6,5564809096439,6,4.26,Port Williamland,2025-08-31T14:31:37.000Z,Debra Smith
7,179856869529778,7,3.67,West Erinborough,2025-09-17T05:57:45.000Z,Justin Peters
8,706321839008097,8,4.9,Lake Stephen,2025-08-26T06:45:51.000Z,Todd Young
9,17279105045499,9,4.5,West Lindsey,2025-08-27T13:04:32.000Z,Mary Young
10,5096134480,10,4.04,Lauraland,2025-09-09T05:50:23.000Z,Jacob Mack


In [0]:
driv_obj = transformations()
df_drivers = driv_obj.dedup(df_drivers,["driver_id"],"last_updated_timestamp")




### _PROCESS DATE_

In [0]:
driv_obj = transformations()
df_drivers = driv_obj.process_timestamp(df_drivers)


In [0]:
if not spark.catalog.tableExists("uber_dbt.silver.drivers"):
  df_drivers.write.mode("append").saveAsTable("uber_dbt.silver.drivers")

else:
    driv_obj.upsert(df_drivers,["driver_id"],"drivers","last_updated_timestamp")

In [0]:
%sql
SELECT COUNT(*) AS Total FROM uber_dbt.silver.drivers

Total
50


# **LOCATIONS**

In [0]:
locations = spark.read.table("uber_dbt.bronze.locations")
display(locations)

location_id,city,state,country,latitude,longitude,last_updated_timestamp
1,Lake Davidport,Hawaii,Dominica,34.5682,151.8097,2025-09-03T13:37:24.000Z
2,Mccarthybury,Minnesota,Dominican Republic,71.6928,-162.2836,2025-09-10T10:54:18.000Z
3,Bellhaven,Arkansas,Japan,-53.3075,69.1475,2025-09-17T06:52:51.000Z
4,Moorechester,New Hampshire,Togo,77.4281,57.0052,2025-08-27T19:04:27.000Z
5,Glennview,North Carolina,Gambia,41.1122,-162.4836,2025-08-20T17:31:06.000Z
6,Davilaville,Arizona,Antarctica (the territory South of 60 deg S),64.3413,-3.912,2025-09-05T13:22:58.000Z
7,Shawnfurt,New Jersey,Lithuania,71.1198,-143.6791,2025-09-03T20:35:10.000Z
8,Bradleytown,Florida,Monaco,30.0895,88.091,2025-08-29T19:45:05.000Z
9,East Miguel,Maryland,French Guiana,-52.9274,-56.9343,2025-08-27T23:43:41.000Z
10,Masonside,Wyoming,Saint Lucia,84.9905,81.5852,2025-09-06T20:25:27.000Z


In [0]:
df_loc = transformations()
drivers = df_loc.dedup(locations,["location_id"],"last_updated_timestamp")

In [0]:
df_loc = transformations()
if not spark.catalog.tableExists("uber_dbt.silver.locations"):
  locations.write.mode("append").saveAsTable("uber_dbt.silver.locations")

else:
    df_loc.upsert(locations,["location_id"],"locations","last_updated_timestamp")

In [0]:
%sql
SELECT count(*) as total FROM uber_dbt.silver.locations

total
50


## **PAYMENTS**

In [0]:
payments = spark.read.table("uber_dbt.bronze.payments")
display(payments)

payment_id,trip_id,customer_id,payment_method,payment_status,amount,transaction_time,last_updated_timestamp
1,274,126,Cash,Success,38.15,2025-09-17T13:00:12.000Z,2025-08-30T13:40:53.000Z
2,676,131,Cash,Success,52.07,2025-08-14T13:00:12.000Z,2025-09-08T18:21:05.000Z
3,919,132,Card,Failed,55.5,2025-07-27T13:00:12.000Z,2025-08-21T20:24:08.000Z
4,247,34,Wallet,Pending,28.78,2025-07-27T13:00:12.000Z,2025-08-27T15:03:09.000Z
5,386,62,Card,Failed,55.02,2025-08-01T13:00:12.000Z,2025-09-08T23:15:06.000Z
6,834,78,Wallet,Success,27.43,2025-08-25T13:00:12.000Z,2025-09-20T09:03:05.000Z
7,348,144,Wallet,Failed,74.94,2025-08-26T13:00:12.000Z,2025-08-30T19:11:26.000Z
8,558,82,Card,Failed,19.58,2025-09-09T13:00:12.000Z,2025-09-06T05:53:26.000Z
9,260,151,Card,Pending,24.4,2025-07-23T13:00:12.000Z,2025-09-02T17:43:14.000Z
10,133,70,Cash,Failed,69.99,2025-07-31T13:00:12.000Z,2025-09-11T10:30:16.000Z


In [0]:
payments = payments.withColumn("online_payment_status",
                               when((col("payment_method") == 'Card') & (col("payment_status") == 'Success'), 'online_success')
                               .when((col("payment_method") == 'Card') & (col("payment_status") == 'Failed'), 'online_failed')
                               .when((col("payment_method") == 'Card') & (col("payment_status") == 'Pending'), 'online_pending')
                               .otherwise('offline'))
display(payments)


payment_id,trip_id,customer_id,payment_method,payment_status,amount,transaction_time,last_updated_timestamp,online_payment_status
1,274,126,Cash,Success,38.15,2025-09-17T13:00:12.000Z,2025-08-30T13:40:53.000Z,offline
2,676,131,Cash,Success,52.07,2025-08-14T13:00:12.000Z,2025-09-08T18:21:05.000Z,offline
3,919,132,Card,Failed,55.5,2025-07-27T13:00:12.000Z,2025-08-21T20:24:08.000Z,online_failed
4,247,34,Wallet,Pending,28.78,2025-07-27T13:00:12.000Z,2025-08-27T15:03:09.000Z,offline
5,386,62,Card,Failed,55.02,2025-08-01T13:00:12.000Z,2025-09-08T23:15:06.000Z,online_failed
6,834,78,Wallet,Success,27.43,2025-08-25T13:00:12.000Z,2025-09-20T09:03:05.000Z,offline
7,348,144,Wallet,Failed,74.94,2025-08-26T13:00:12.000Z,2025-08-30T19:11:26.000Z,offline
8,558,82,Card,Failed,19.58,2025-09-09T13:00:12.000Z,2025-09-06T05:53:26.000Z,online_failed
9,260,151,Card,Pending,24.4,2025-07-23T13:00:12.000Z,2025-09-02T17:43:14.000Z,online_pending
10,133,70,Cash,Failed,69.99,2025-07-31T13:00:12.000Z,2025-09-11T10:30:16.000Z,offline


In [0]:
pay_obj = transformations()
payments = pay_obj.dedup(payments,["payment_id"],"last_updated_timestamp")
payments = pay_obj.process_timestamp(payments)
if not spark.catalog.tableExists("uber_dbt.silver.payments"):
  payments.write.mode("append").saveAsTable("uber_dbt.silver.payments")

else:
    pay_obj.upsert(payments,["payment_id"],"payments","last_updated_timestamp")


In [0]:
%sql
SELECT count(*) as total FROM uber_dbt.silver.payments

total
1000


## **VEHICLES**

In [0]:
vech = spark.read.table("uber_dbt.bronze.vehicles")
display(vech)


vehicle_id,license_plate,model,make,year,vehicle_type,last_updated_timestamp
1,NXT-8646,Message,"Francis, Smith and Lee",2023,Hatchback,2025-09-02T06:05:20.000Z
2,03S R43,Region,Lawson Group,2017,Sedan,2025-08-31T05:55:09.000Z
3,SKO H06,Prepare,"Moreno, Ruiz and Barker",2023,Luxury,2025-08-30T05:07:29.000Z
4,5R235,Pattern,"Welch, Martinez and Hendricks",2019,Van,2025-09-09T05:52:10.000Z
5,925A,Process,Rivera-Anderson,2014,Hatchback,2025-08-27T11:42:16.000Z
6,6-1130V,On,Brown Ltd,2014,SUV,2025-09-06T06:37:52.000Z
7,L25-TWN,Plan,"Gonzalez, Rios and Rios",2020,Sedan,2025-09-15T13:14:48.000Z
8,GWY 958,Month,"Smith, Mckenzie and Bullock",2017,Sedan,2025-09-11T16:20:12.000Z
9,4FG 919,Speak,"Rice, Barnes and Hernandez",2019,SUV,2025-09-11T02:43:13.000Z
10,6FD 648,Religious,Schwartz and Sons,2017,SUV,2025-09-11T19:10:01.000Z


In [0]:
vech = vech.withColumn("make",upper(col("make")))
display(vech)

vehicle_id,license_plate,model,make,year,vehicle_type,last_updated_timestamp
1,NXT-8646,Message,"FRANCIS, SMITH AND LEE",2023,Hatchback,2025-09-02T06:05:20.000Z
2,03S R43,Region,LAWSON GROUP,2017,Sedan,2025-08-31T05:55:09.000Z
3,SKO H06,Prepare,"MORENO, RUIZ AND BARKER",2023,Luxury,2025-08-30T05:07:29.000Z
4,5R235,Pattern,"WELCH, MARTINEZ AND HENDRICKS",2019,Van,2025-09-09T05:52:10.000Z
5,925A,Process,RIVERA-ANDERSON,2014,Hatchback,2025-08-27T11:42:16.000Z
6,6-1130V,On,BROWN LTD,2014,SUV,2025-09-06T06:37:52.000Z
7,L25-TWN,Plan,"GONZALEZ, RIOS AND RIOS",2020,Sedan,2025-09-15T13:14:48.000Z
8,GWY 958,Month,"SMITH, MCKENZIE AND BULLOCK",2017,Sedan,2025-09-11T16:20:12.000Z
9,4FG 919,Speak,"RICE, BARNES AND HERNANDEZ",2019,SUV,2025-09-11T02:43:13.000Z
10,6FD 648,Religious,SCHWARTZ AND SONS,2017,SUV,2025-09-11T19:10:01.000Z


In [0]:
vech_obj = transformations()

vech = vech_obj.dedup(vech,["vehicle_id"],"last_updated_timestamp")

vech = vech_obj.process_timestamp(vech)

if not spark.catalog.tableExists("uber_dbt.silver.vehicles"):
  vech.write.mode("append").saveAsTable("uber_dbt.silver.vehicles")

else:
    vech_obj.upsert(vech,["vehicle_id"],"vehicles","last_updated_timestamp")

In [0]:
%sql
SELECT COUNT(*) FROM uber_dbt.silver.vehicles

COUNT(*)
50


# **DATA BUILD TOOL (DBT)**

### _TRANSFORMATION FACT TABLE WITH DBT_

## **TRIPS**

In [0]:
trips = spark.read.table("uber_dbt.bronze.trips")
display(trips)

trip_id,driver_id,customer_id,vehicle_id,trip_start_time,trip_end_time,start_location,end_location,distance_km,fare_amount,payment_method,trip_status,last_updated_timestamp
1,29,185,29,2025-07-31T18:00:11.000Z,2025-07-31T18:07:11.000Z,New Victoriahaven,New Cynthiamouth,9.09,28.41,Card,Cancelled,2025-09-06T01:55:37.000Z
2,2,180,2,2025-08-29T04:00:11.000Z,2025-08-29T04:36:11.000Z,West Wanda,Donaldburgh,22.68,76.3,Card,Ongoing,2025-08-25T21:45:53.000Z
3,43,182,43,2025-08-25T09:00:11.000Z,2025-08-25T09:36:11.000Z,North Robert,West Donald,29.23,47.0,Cash,Completed,2025-09-07T12:33:38.000Z
4,4,152,4,2025-08-10T21:00:11.000Z,2025-08-10T21:16:11.000Z,Chadville,West Marcusmouth,25.07,35.69,Wallet,Completed,2025-09-04T10:41:23.000Z
5,23,132,23,2025-09-12T15:00:11.000Z,2025-09-12T15:37:11.000Z,Schneiderview,Ochoaton,33.2,49.46,Card,Ongoing,2025-09-20T04:33:10.000Z
6,31,80,31,2025-07-11T06:00:11.000Z,2025-07-11T06:59:11.000Z,Williamsbury,Monroemouth,7.46,21.01,Wallet,Completed,2025-08-23T06:21:23.000Z
7,15,172,15,2025-07-12T06:00:11.000Z,2025-07-12T06:31:11.000Z,New Dana,Carpenterport,36.28,128.96,Cash,Completed,2025-09-05T17:04:37.000Z
8,4,12,4,2025-07-27T07:00:11.000Z,2025-07-27T07:14:11.000Z,Whitneybury,West Kennethmouth,33.88,57.18,Wallet,Ongoing,2025-08-22T14:35:00.000Z
9,27,160,27,2025-07-03T16:00:11.000Z,2025-07-03T16:05:11.000Z,West Christopherhaven,Nicholasberg,16.03,51.53,Wallet,Ongoing,2025-09-15T09:56:31.000Z
10,41,126,41,2025-09-05T05:00:11.000Z,2025-09-05T05:20:11.000Z,Ashleyborough,North Morganburgh,20.53,42.27,Card,Cancelled,2025-08-22T12:08:26.000Z
